# CS-340 Project Two — Grazioso Salvare Dashboard
**Abraham Kollie Jr.**

Run the code cell below in Codio after setting the MongoDB password. Keep `CRUD_Python_Module.py` and the Grazioso Salvare logo in the same folder.

In [1]:
# CS-340 Project Two | Grazioso Salvare Dashboard
# Abraham Kollie Jr.
import os
import base64
import pandas as pd
import plotly.express as px
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
from jupyter_dash import JupyterDash
from CRUD_Python_Module import AnimalShelter

# Enter the password for your existing MongoDB aacuser account below.
# Do not upload real passwords to a public GitHub repository.
username = "aacuser"
password = "Ckollie.20"
if password.startswith("REPLACE_"):
    raise ValueError("Replace password with the working aacuser MongoDB password before running.")
shelter = AnimalShelter(username, password)

# Match the preferred breeds, sex, and training ages in the dashboard specifications.
FILTERS = {
    'water': (['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland'], 'Intact Female', 26, 156),
    'mountain': (['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler'], 'Intact Male', 26, 156),
    'disaster': (['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler'], 'Intact Male', 20, 300)
}

def rescue_query(kind):
    if kind == 'reset':
        return {}
    breeds, sex, min_weeks, max_weeks = FILTERS[kind]
    # Includes breed variants such as German Shepherd Mix.
    breed_conditions = [{'breed': {'$regex': '^' + __import__('re').escape(b), '$options': 'i'}} for b in breeds]
    return {'animal_type': 'Dog', '$or': breed_conditions,
            'sex_upon_outcome': sex,
            'age_upon_outcome_in_weeks': {'$gte': min_weeks, '$lte': max_weeks}}

def fetch_animals(kind):
    records = shelter.read(rescue_query(kind))
    df = pd.DataFrame.from_records(records)
    if '_id' in df.columns:
        df = df.drop(columns=['_id'])
    for column in ('location_lat', 'location_long'):
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors='coerce')
    return df

# Verify the database works before building the dashboard.
initial_df = fetch_animals('reset')
if initial_df.empty:
    raise RuntimeError('MongoDB returned no records. Check the aacuser password, MongoDB service, and aac.animals collection.')

# Use the logo already supplied in the Codio code_files directory.
logo_candidates = ['Grazioso Salvare Logo.png', 'Grazioso Salvare Logo.jpg',
                   'Grazioso Salvare Logo.jpeg', 'Grazioso Salvare Logo.webp']
logo_path = next((p for p in logo_candidates if os.path.isfile(p)), None)
if logo_path is None:
    logo_path = next((p for p in os.listdir('.') if 'grazioso' in p.lower() and p.lower().endswith(('.png','.jpg','.jpeg','.webp'))), None)
logo = None
if logo_path:
    ext = logo_path.rsplit('.', 1)[-1].lower()
    mime = 'jpeg' if ext in ('jpg', 'jpeg') else ext
    with open(logo_path, 'rb') as f:
        logo = 'data:image/' + mime + ';base64,' + base64.b64encode(f.read()).decode('ascii')

app = JupyterDash(__name__)
app.layout = html.Div([
    html.Div([
        html.A(html.Img(src=logo, style={'height':'100px', 'maxWidth':'230px', 'objectFit':'contain'}) if logo else 'Grazioso Salvare (logo file not found)',
               href='https://www.snhu.edu', target='_blank'),
        html.Div([html.H1('SNHU CS-340 Rescue Animal Dashboard', style={'marginBottom':'4px'}),
                  html.B('Created by Abraham Kollie Jr. | AKOLLIE-JR')])
    ], style={'display':'flex','alignItems':'center','gap':'30px','padding':'14px'}),
    html.H3('Select rescue training category'),
    dcc.RadioItems(id='rescue-filter', value='reset',
        options=[{'label':' Reset (All Animals)', 'value':'reset'},
                 {'label':' Water Rescue', 'value':'water'},
                 {'label':' Mountain or Wilderness Rescue', 'value':'mountain'},
                 {'label':' Disaster or Individual Tracking', 'value':'disaster'}],
        labelStyle={'display':'inline-block','marginRight':'22px','marginBottom':'12px'}),
    html.Div(id='results-count', style={'margin':'8px 0','fontWeight':'bold'}),
    dash_table.DataTable(id='datatable-id',
        columns=[{'name':c,'id':c} for c in initial_df.columns],
        data=initial_df.to_dict('records'), page_size=10, page_action='native',
        sort_action='native', filter_action='native', row_selectable='single',
        selected_rows=[], style_table={'overflowX':'auto'},
        style_cell={'textAlign':'left','padding':'7px','minWidth':'110px','maxWidth':'260px','overflow':'hidden','textOverflow':'ellipsis'},
        style_header={'fontWeight':'bold','backgroundColor':'#edf2f7'},
        style_data_conditional=[{'if':{'state':'selected'},'backgroundColor':'#dbeafe'}]),
    html.Div([
        html.Div([html.H3('Breed Distribution'), dcc.Graph(id='breed-chart')],
                 style={'flex':'1','minWidth':'350px'}),
        html.Div([html.H3('Animal Geolocation'), html.Div(id='map-id')],
                 style={'flex':'1','minWidth':'350px'})
    ], style={'display':'flex','gap':'20px','flexWrap':'wrap','marginTop':'16px'})
], style={'fontFamily':'Arial, sans-serif','padding':'18px','maxWidth':'1600px','margin':'auto'})

@app.callback([Output('datatable-id','data'), Output('datatable-id','columns'),
               Output('results-count','children'), Output('datatable-id','selected_rows')],
              [Input('rescue-filter','value')])
def update_table(kind):
    df = fetch_animals(kind)
    return (df.to_dict('records'), [{'name':c,'id':c} for c in df.columns],
            f'{len(df):,} matching animal records', [])

@app.callback([Output('breed-chart','figure'), Output('map-id','children')],
              [Input('datatable-id','derived_virtual_data'),
               Input('datatable-id','derived_virtual_selected_rows'),
               Input('datatable-id','data')])
def update_charts(visible, selected, raw):
    df = pd.DataFrame(visible if visible is not None else (raw or []))
    if df.empty or 'breed' not in df.columns:
        fig = px.pie(names=['No matching animals'], values=[1], title='No matching animals')
        return fig, html.P('No locations available for this filter.')
    counts = df['breed'].fillna('Unknown').value_counts().head(10)
    fig = px.pie(names=counts.index.tolist(), values=counts.values.tolist(), title='Top 10 Breeds')
    fig.update_layout(margin=dict(l=10,r=10,t=50,b=10))
    geo = df.copy()
    for col in ('location_lat','location_long'):
        if col not in geo.columns:
            return fig, html.P('Location fields are missing from the database.')
        geo[col] = pd.to_numeric(geo[col], errors='coerce')
    geo = geo.dropna(subset=['location_lat','location_long'])
    geo = geo[geo['location_lat'].between(-90,90) & geo['location_long'].between(-180,180)]
    if geo.empty:
        return fig, html.P('No valid geolocation coordinates available.')
    # Show a single selected animal, otherwise show up to 100 animals for responsiveness.
    if selected and 0 <= selected[0] < len(df):
        chosen = df.iloc[selected[0]]
        if pd.notna(pd.to_numeric(chosen.get('location_lat'), errors='coerce')) and pd.notna(pd.to_numeric(chosen.get('location_long'), errors='coerce')):
            geo = pd.DataFrame([chosen])
    else:
        geo = geo.head(100)
    markers = []
    for _, animal in geo.iterrows():
        markers.append(dl.Marker(position=[float(animal['location_lat']),float(animal['location_long'])],
            children=[dl.Tooltip(str(animal.get('breed','Animal'))),
                      dl.Popup(html.Div([html.B(str(animal.get('name') or 'Unnamed')),
                                         html.P(str(animal.get('breed','Unknown breed'))),
                                         html.P('ID: ' + str(animal.get('animal_id','N/A')))]))]))
    center = [float(geo.iloc[0]['location_lat']), float(geo.iloc[0]['location_long'])] if len(geo)==1 else [30.2672,-97.7431]
    return fig, dl.Map(children=[dl.TileLayer(), *markers], center=center,
                       zoom=11 if len(geo)==1 else 9,
                       style={'width':'100%','height':'450px'})

# If the proxy link opens but dashboard does not display, try mode='external'.
app.run_server(
    mode="external",
    host="0.0.0.0",
    port=8050,
    debug=False
)


 * Running on all addresses.
 * Running on http://10.15.164.95:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [12/Oct/2026 20:25:21] "GET /_alive_7d77bdaa-2218-46f2-9792-ee642df047d2 HTTP/1.1" 200 -


Dash app running on http://0.0.0.0:8050/
